# AIE S3 — Diabetes Progression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s3-diabetes-progression.ipynb)

**Regression.** Predict disease progression one year after baseline,
from ten clinical measurements.

Challenge: <https://ml-arena.com/viewchallenge/185>

---

This notebook exists to demonstrate one claim, on real data, with
numbers you can re-run:

> **Your training score is not your score.**

We fit two models — a plain straight line and a 500-tree random
forest — and the forest will look about twice as good. It is not.
By the end you will have seen three ways to find that out, ordered
from worst to best: the training score (useless), a single held-out
split (better, but noisy), and cross-validation (what to actually
use). Then we submit both models and let the leaderboard confirm it.

---

## 0. Setup

The distribution is **`mlarena-sdk`** and it imports as `mlarena`.
`pip install mlarena` is an unrelated package.

In [ ]:
!pip install -q mlarena-sdk

---

## 1. Get the data

In [ ]:
import mlarena

API_KEY = "mlk_user_..."   # <- paste yours here
CHALLENGE_ID = 185

client = mlarena.connect(api_key=API_KEY)
client.download_dataset(CHALLENGE_ID, ".")

---

## 2. Read it

In [ ]:
import pandas as pd

X = pd.read_csv("X.csv")
y = pd.read_csv("y.csv")["prediction"]
X_submission = pd.read_csv("X_submission.csv")

FEATURES = [c for c in X.columns if c != "id"]
print("X", X.shape, " X_submission", X_submission.shape)
X.head()

**n = 265 training rows, p = 10 features.** That is a small dataset,
and it is small on purpose: the fewer rows you have, the easier it is
for a flexible model to memorise them.

The target is a continuous progression score. All ten features are
numeric, so there is nothing to encode — this notebook is about
validation, not preparation.

In [ ]:
print(y.describe().round(1))
X[FEATURES].describe().round(1)

---

## 3. A quick look

Two plots, enough to know what we are working with.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = X[FEATURES].assign(target=y)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
sns.histplot(df["target"], bins=30, ax=axes[0], color="#2f6f9f")
axes[0].set_title("Disease progression at one year")

corr = df.corr(numeric_only=True)["target"].drop("target").sort_values()
corr.plot.barh(ax=axes[1], color="#2f6f9f")
axes[1].set_title("Correlation of each feature with the target")
plt.tight_layout()
plt.show()

`bmi` and `log_triglycerides` carry the most linear signal, `hdl` the
most negative. Nothing here is overwhelming — this is a genuinely
noisy problem, which is the second reason it was chosen.

---

## 4. The trap

Fit both models on the training data, and score them on **the same**
training data. This is the thing you must not do, and it is worth
doing once so you recognise the result when you see it in the wild.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

linear = LinearRegression().fit(X[FEATURES], y)
forest = RandomForestRegressor(n_estimators=500, random_state=0,
                               n_jobs=-1).fit(X[FEATURES], y)

print(f"LinearRegression   train R2 = {r2_score(y, linear.predict(X[FEATURES])):.4f}")
print(f"RandomForest(500)  train R2 = {r2_score(y, forest.predict(X[FEATURES])):.4f}")

**0.51 against 0.92.** The forest explains 92% of the variance and the
line barely half. On this evidence the forest is not slightly better,
it is in a different league, and the obvious move is to ship it.

This evidence is worthless. The forest was *shown the answers* for
all 265 rows and is now being asked about those same 265 rows. It has
500 trees grown until their leaves are pure — it has enough capacity
to store the training set, so a high score here measures storage
capacity, not skill.

---

## 5. Hold some data back

The cheapest honest test: fit on part of the data, score on the part
the model never saw.

In [ ]:
from sklearn.model_selection import train_test_split

X_fit, X_val, y_fit, y_val = train_test_split(
    X[FEATURES], y, test_size=0.25, random_state=0)
print(f"fit on {len(X_fit)} rows, validate on {len(X_val)}")
print()

for label, model in [("LinearRegression", LinearRegression()),
                     ("RandomForest(500)", RandomForestRegressor(
                         n_estimators=500, random_state=0, n_jobs=-1))]:
    model.fit(X_fit, y_fit)
    tr = r2_score(y_fit, model.predict(X_fit))
    va = r2_score(y_val, model.predict(X_val))
    print(f"{label:18} train {tr:.4f}   validation {va:.4f}   gap {tr - va:+.4f}")

There it is. The line loses about 0.19 between what it fitted and
what it generalises; the forest loses **0.64**, and ends up *below*
the line on the rows that count.

That difference between the two columns is what the word
**overfitting** names. It is not a property of random forests — it is
what happens when a model has more capacity than the data can
constrain.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = ["LinearRegression", "RandomForest(500)"]
scores = {}
for label, model in zip(labels, [LinearRegression(),
        RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=-1)]):
    model.fit(X_fit, y_fit)
    scores[label] = (r2_score(y_fit, model.predict(X_fit)),
                     r2_score(y_val, model.predict(X_val)))

import numpy as np
pos = np.arange(len(labels))
ax.bar(pos - 0.2, [scores[l][0] for l in labels], 0.4, label="train", color="#c1553b")
ax.bar(pos + 0.2, [scores[l][1] for l in labels], 0.4, label="validation", color="#2f6f9f")
ax.set_xticks(pos); ax.set_xticklabels(labels)
ax.set_ylabel("R²"); ax.legend()
ax.set_title("The gap between the two bars is the overfitting")
plt.show()

---

## 6. One split is not enough

The validation set above had 67 rows. Change `random_state` and the
numbers move a lot — you are estimating a score from 67 draws. Before
trusting any single split, look at how much it wobbles.

In [ ]:
for seed in range(5):
    A, B, a, b = train_test_split(X[FEATURES], y, test_size=0.25, random_state=seed)
    lin = LinearRegression().fit(A, a)
    rf = RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=-1).fit(A, a)
    print(f"seed {seed}:  linear {r2_score(b, lin.predict(B)):.3f}   forest {r2_score(b, rf.predict(B)):.3f}")

The individual numbers swing by more than the difference between the
two models. A single split can hand you almost any conclusion,
including the wrong one.

---

## 7. Cross-validation

Instead of holding out one block, hold out each of five in turn: fit
on the other four, score on the held-out fold, repeat. Every row gets
used for training four times and for scoring once, and you end up
with five estimates instead of one.

`cross_val_score` does the whole loop.

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

for label, model in [("LinearRegression", LinearRegression()),
                     ("RandomForest(500)", RandomForestRegressor(
                         n_estimators=500, random_state=0, n_jobs=-1))]:
    cv = cross_val_score(model, X[FEATURES], y, cv=5, scoring="r2")
    print(f"{label:18} CV R2 = {cv.mean():.4f} +/- {cv.std():.4f}   folds {np.round(cv, 3)}")

Two things to read here.

**The mean settles the argument.** 0.427 for the line against 0.358
for the forest — the line generalises better, and we established that
without touching a single test label.

**The spread explains section 6.** The individual folds run from about
0.27 to 0.63 for the line alone. That range *is* the noise a single
split was sampling from. Reporting one number from one split is
reporting one draw from that distribution and calling it the answer.

This is the number to make decisions on.

---

## 8. The forest is not the problem — its settings are

An unrestricted tree grows until every leaf is pure. Force the leaves
to hold at least `min_samples_leaf` rows and the model can no longer
memorise individuals. Watch both curves as we turn that dial.

In [ ]:
leaves = [1, 2, 5, 10, 20, 40]
train_scores, cv_scores = [], []
for leaf in leaves:
    rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=leaf,
                               random_state=0, n_jobs=-1)
    train_scores.append(r2_score(y, rf.fit(X[FEATURES], y).predict(X[FEATURES])))
    cv_scores.append(cross_val_score(rf, X[FEATURES], y, cv=5, scoring="r2").mean())

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(leaves, train_scores, "o-", color="#c1553b", label="train R²")
ax.plot(leaves, cv_scores, "o-", color="#2f6f9f", label="5-fold CV R²")
ax.axhline(0.4273, ls="--", c="gray", lw=1, label="LinearRegression CV")
ax.set_xscale("log"); ax.set_xticks(leaves)
ax.set_xticklabels(leaves)
ax.set_xlabel("min_samples_leaf  (less flexible ->)")
ax.set_ylabel("R²"); ax.legend()
ax.set_title("Constraining the forest costs training score and buys generalisation")
plt.show()

for leaf, tr, cv in zip(leaves, train_scores, cv_scores):
    print(f"min_samples_leaf={leaf:<3} train {tr:.4f}   CV {cv:.4f}   gap {tr - cv:+.4f}")

The red curve falls the whole way — every constraint costs training
score. The blue curve *rises* to a peak around `min_samples_leaf=10`
and then falls again: too little capacity is its own failure.

This is the bias–variance tradeoff, drawn from your own data. The
gap between the curves shrinks from 0.56 to 0.09. And note where the
blue curve peaks relative to the dashed line — even tuned, the forest
does not overtake the straight line on this dataset. Sometimes the
simple model is simply right.

---

## 9. Predict the submission set and submit both

Refit both on all 265 rows and write two submissions. We submit the
forest as well as the line, on purpose: the leaderboard is an
independent referee and you should watch it agree with the CV.

In [ ]:
linear = LinearRegression().fit(X[FEATURES], y)
forest = RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=-1).fit(X[FEATURES], y)

for fname, model in [("submission.csv", linear), ("forest.csv", forest)]:
    pd.DataFrame({"id": X_submission["id"],
                  "prediction": model.predict(X_submission[FEATURES])}).to_csv(fname, index=False)
    print(f"wrote {fname}")

In [ ]:
assert len(pd.read_csv("submission.csv")) == len(X_submission)
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"],
                       agent_name="linear-regression")
print(result)

Now the forest. `submit` uploads each file under its basename and the
challenge expects `submission.csv`, so copy it to that name first.

In [ ]:
import shutil
shutil.copy("forest.csv", "submission.csv")
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"],
                       agent_name="random-forest-500")
print(result)

Scoring takes a moment. Then look at the board:

In [ ]:
client.leaderboard(CHALLENGE_ID)

---

## 10. What to take away

The challenge page lists the full ladder. Two facts from it are worth
carrying out of this session:

- Of six models tried on this split, **the two with the best training
  score have the worst test score**, and the straight line — last on
  training — is first on test.
- Rank those six by 5-fold CV on the training set alone and you
  recover the test ranking **exactly**, without ever seeing a test
  label.

So the working rule for the rest of the course:

> Choose models on cross-validation. Use the held-out set once, to
> confirm.

And the corollary people forget: **the leaderboard is a test set too.**
Submit twenty variants and keep the best, and you have fitted 177 rows
by hand — the same mistake, one level up. Your CV score is the
estimate you should believe.